In [ ]:
# 1. Variáveis extraídas da sua nova URL do Pooler
host = "aws-1-sa-east-1.pooler.supabase.com"
port = "5432"
database = "postgres"
user = "postgres.bpiwbiwzoybrpdjjfbyn" # Atenção: O usuário agora tem o ID do projeto!
password = "rh@datamesh"

# 2. Montando a URL no formato JDBC
jdbc_url = f"jdbc:postgresql://{host}:{port}/{database}"

# 3. Propriedades da conexão
connection_properties = {
    "user": user,
    "password": password,
    "driver": "org.postgresql.Driver",
    "ssl": "true",
    "sslmode": "require"
}

# 4. Lendo os dados
# Criamos uma função que recebe o nome da tabela
def extrair_tabela_supabase(nome_da_tabela):
    print(f"Extraindo tabela: {nome_da_tabela}...")
    
    df = spark.read.jdbc(
        url=jdbc_url, 
        table=nome_da_tabela, 
        properties=connection_properties
    )
    return df

In [ ]:
# Lista com as tabelas que você quer trazer do Supabase
tabelas_para_ingestao = [
    "colaborador",
    "departamento",
    "ausencia",
    "cargo",
    "movimentacao_colaborador",
    "candidatura",
    "recrutamento_vaga"
]

# O loop que vai processar uma por uma
for tabela in tabelas_para_ingestao:
    try:
        # 1. Usa a função para ler os dados do Supabase
        df_tabela = extrair_tabela_supabase(tabela)
        
        # 2. Salva os dados dentro do Databricks (formato Delta)
        # Estamos adicionando o prefixo "bronze_" para indicar que são dados brutos
        nome_tabela_destino = f"bronze_{tabela}"
        
        df_tabela.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(nome_tabela_destino)
            
        print(f"✅ Tabela '{tabela}' salva com sucesso como '{nome_tabela_destino}'!")
        
    except Exception as e:
        print(f"❌ Erro ao processar a tabela '{tabela}': {e}")